#### Import Necessary Libraries

In [ ]:
import pandas as pd
import bz2
import os
import glob
from tqdm import tqdm
from langdetect import detect

In [ ]:
def check_language(text):
    """
    Check if the tweet is in English language
    
    Parameters:
    text: str
        The tweet text
        
    Returns:
    str
        The tweet text if it is in English language
    None
        If the tweet is not in English language
    """
    
    try:
        if detect(text) == 'en':
            return text
    except:
        return None

In [ ]:
def processDataFolder(folderPath):
    """
        Process the data in the folder and create a csv file
        
        Parameters:
        folderPath: str
            The folder path containing the data
            
        Returns:
        None
    """
    # Folder path
    main_folder_path = '../ProcessedData/'+folderPath
    print(main_folder_path)
    
    # Use os.listdir to get a list of all subfolders in the main folder
    subfolders = [f.path for f in os.scandir(main_folder_path) if f.is_dir()]

    # Initialize an empty list to store individual DataFrames
    dfs = []

    # Loop through each subfolder
    for subfolder in tqdm(subfolders):
        # Use glob to get a list of all JSON files in the subfolder
        json_files = glob.glob(subfolder + '/*.json.bz2')
        
        # Loop through each JSON file in the subfolder
        for json_file in json_files:
            with bz2.open(json_file, 'rt', encoding='utf-8') as file:
                # Read the JSON file into a DataFrame
                df = pd.read_json(file, lines=True)
                df = df[['text', 'created_at', 'user']]
                df = df.dropna(subset=['text'])
                df = df.drop_duplicates(subset=['text'])
                df['cleaned_tweet'] = df['text'].apply(check_language)
                df.dropna(subset=['cleaned_tweet'], inplace=True)
                # Append the DataFrame to the list
                dfs.append(df)
    
    # Csv file path
    csv_file = folderPath+'.csv'
    
    # Concatenate all DataFrames into a single DataFrame
    final_df = pd.concat(dfs, ignore_index=True)
    final_df.shape
    final_df.to_csv(csv_file, index=False)
    print('created csv file: ',csv_file)

In [ ]:
# Looping through the folders of the days of week 3
week3_days = ['15','16','17','18','19','20','21']
for day in week3_days:
    processDataFolder(day)

In [ ]:
# Looping through the csv files of the days of a week
# Loading the csv files of the days of the week into a single dataframe

# Path to the folder containing the CSV files
folder_path = './'

# Get a list of all CSV files in the folder
csv_files = [file for file in os.listdir(folder_path) if file.endswith('.csv')]

# Initialize an empty list to store DataFrames
all_dfs = []

# Iterate over each CSV file, load it, and append to the list of DataFrames
for file in csv_files:
    file_path = os.path.join(folder_path, file)
    print(file_path)
    df = pd.read_csv(file_path, lineterminator='\n')
    all_dfs.append(df) 

# Concatenate all DataFrames into one
combined_df = pd.concat(all_dfs, ignore_index=True)

# Optionally, you can reset the index if needed
combined_df.reset_index(drop=True, inplace=True)

# Display the combined DataFrame
print(combined_df.shape)
combined_df.head()

In [ ]:
# Create a CSV file from the combined DataFrame
combined_df[['text', 'created_at', 'user']].to_csv('Week3.csv', index=False)